# Recommendation Baselines — Phase 3

This notebook implements Phase 3 steps 1–5: final artifact-contract validation, task definitions, candidate policies and ranking metrics. It deliberately does not implement recommendation baselines yet; those begin at step 6.

> Validation-future data is evaluation-only. Discovery-future labels may be used for model training in later steps, but no validation-future label may influence fitting or tuning.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1 — Imports, paths and immutable experiment configuration

In [2]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

BENCHMARK_ROOT = Path('/content/drive/MyDrive/datasets/recommendation_benchmark_final_outputs')
OUTPUT_ROOT = Path('/content/drive/MyDrive/datasets/recommendation_baseline_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_DIRS = {
    'ASSISTments': BENCHMARK_ROOT / 'assistments',
    'KDD': BENCHMARK_ROOT / 'kdd',
}
METRIC_KS = (5, 10, 20)
PRIMARY_RELEVANCE = 'attempted'
RANDOM_STATE = 42

ROOT_FILES = [
    'benchmark_summary.csv', 'artifact_manifest.csv',
    'benchmark_config.json', 'benchmark_schema.json',
]
for filename in ROOT_FILES:
    path = BENCHMARK_ROOT / filename


## Step 2 — Validate the 18-artifact benchmark contract

This validation uses Parquet metadata wherever possible so the four-million-row ASSISTments histories are not loaded merely to check their schemas. Small learner and catalogue tables are loaded for semantic assertions.

In [3]:
with open(BENCHMARK_ROOT / 'benchmark_config.json', encoding='utf-8') as file:
    benchmark_config = json.load(file)
with open(BENCHMARK_ROOT / 'benchmark_schema.json', encoding='utf-8') as file:
    benchmark_schema = json.load(file)
benchmark_summary = pd.read_csv(BENCHMARK_ROOT / 'benchmark_summary.csv')
phase2_manifest = pd.read_csv(BENCHMARK_ROOT / 'artifact_manifest.csv')

assert benchmark_config['candidate_statistics_source'] == 'discovery_early_only'
assert benchmark_config['future_role'] == 'relevance_labels_only'
assert benchmark_config['assist_skill_name_mapping_source'] == 'discovery_early_only'
assert len(benchmark_schema) == 9
assert len(phase2_manifest) == 18
assert set(phase2_manifest['Dataset']) == set(DATASET_DIRS)

REQUIRED_COLUMNS = {
    'learner_splits.parquet': {
        'learner_id', 'cohort', 'evaluable_skill', 'evaluable_problem',
        'cold_start_skill_history', 'cold_start_problem_history',
        'cluster', 'cluster_selection_status', 'cluster_allowed_as_primary',
    },
    'skill_catalog.parquet': {
        'item_id', 'item_type', 'training_interactions', 'training_learners',
        'training_success_rate', 'training_popularity', 'catalog_source',
    },
    'problem_catalog.parquet': {
        'item_id', 'item_type', 'training_interactions', 'training_learners',
        'training_success_rate', 'training_popularity', 'catalog_source',
        'problem_type', 'hierarchy',
    },
    'early_skill_history.parquet': {
        'learner_id', 'item_id', 'early_interaction_count',
        'empirical_bayes_mastery', 'mastery_evidence_confidence',
        'skill_state', 'in_candidate_catalog',
    },
    'future_skill_relevance.parquet': {
        'learner_id', 'item_id', 'relevance_binary',
        'successful_future_item', 'in_candidate_catalog', 'seen_in_early',
    },
    'early_problem_history.parquet': {
        'learner_id', 'item_id', 'early_interaction_count',
        'early_success_rate', 'in_candidate_catalog',
    },
    'future_problem_relevance.parquet': {
        'learner_id', 'item_id', 'relevance_binary',
        'successful_future_item', 'in_candidate_catalog', 'seen_in_early',
    },
    'problem_skill_map.parquet': {
        'problem_item_id', 'skill_item_id', 'training_interactions',
        'association_share', 'mapping_source',
    },
    'skill_name_id_map.parquet': {
        'normalised_skill_name', 'mapped_skill_id', 'training_rows',
        'training_learners', 'distinct_skill_ids', 'mapping_source',
    },
}
assert set(REQUIRED_COLUMNS) == set(benchmark_schema)

contract_rows = []
small_tables = {}
for dataset, directory in DATASET_DIRS.items():
    dataset_manifest = phase2_manifest[phase2_manifest['Dataset'].eq(dataset)].set_index('File')
    assert set(dataset_manifest.index) == set(REQUIRED_COLUMNS)
    for filename, required_columns in REQUIRED_COLUMNS.items():
        path = directory / filename
        parquet_file = pq.ParquetFile(path)
        row_count = parquet_file.metadata.num_rows
        schema_columns = set(parquet_file.schema_arrow.names)
        missing_columns = sorted(required_columns - schema_columns)
        manifest_row = dataset_manifest.loc[filename]
        assert row_count == int(manifest_row['Rows'])
        assert path.stat().st_size == int(manifest_row['Bytes'])
        assert not missing_columns, f'{dataset}/{filename}: {missing_columns}'
        contract_rows.append({
            'Dataset': dataset, 'File': filename, 'Rows': row_count,
            'Bytes': path.stat().st_size, 'SchemaValid': True,
            'ManifestValid': True,
        })

    learners = pd.read_parquet(directory / 'learner_splits.parquet')
    skill_catalog = pd.read_parquet(directory / 'skill_catalog.parquet')
    problem_catalog = pd.read_parquet(directory / 'problem_catalog.parquet')
    problem_skill_map = pd.read_parquet(directory / 'problem_skill_map.parquet')
    skill_name_map = pd.read_parquet(directory / 'skill_name_id_map.parquet')
    assert set(learners['cohort']) == {'discovery', 'validation'}
    assert learners['cluster'].notna().all()
    assert skill_catalog['catalog_source'].eq('discovery_early_only').all()
    assert problem_catalog['catalog_source'].eq('discovery_early_only').all()
    assert problem_skill_map['mapping_source'].eq('discovery_early_only').all()
    if len(skill_name_map):
        assert skill_name_map['mapping_source'].eq('discovery_early_only').all()
    small_tables[dataset] = {
        'learners': learners, 'skill_catalog': skill_catalog,
        'problem_catalog': problem_catalog,
    }

assert small_tables['ASSISTments']['learners']['cluster_selection_status'].eq('accepted').all()
assert small_tables['KDD']['learners']['cluster_selection_status'].eq('exploratory_fallback').all()
assert not small_tables['KDD']['learners']['cluster_allowed_as_primary'].any()
contract_validation = pd.DataFrame(contract_rows)
display(benchmark_summary)
display(contract_validation)
print('Validated 18 Phase 2 Parquet artifacts without loading the large histories.')

,Dataset,RawRows,RawLearners,EligibleLearners,DiscoveryLearners,ValidationLearners,EarlyInteractions,FutureInteractions,SkillCandidates,ProblemCandidates,...,ValidationProblemEvaluableRate,ValidationSkillColdStartRate,ValidationProblemColdStartRate,ClusterSelectionStatus,TrainingSkillNameMappings,AmbiguousTrainingSkillNames,EarlyRowsMappedFromName,FutureRowsMappedFromName,EarlyTextFallbackRows,FutureTextFallbackRows
0,ASSISTments,6117947,46667,33335,26668,6667,4184236,1814850,161,39779,...,0.899355,0.375731,0.024599,accepted,194,33,0,0,0,0
1,KDD,809694,574,565,452,113,566460,243152,99,855,...,0.982301,0.000000,0.000000,exploratory_fallback,0,0,0,0,0,0


,Dataset,File,Rows,Bytes,SchemaValid,ManifestValid
0,ASSISTments,learner_splits.parquet,33335,1283461,True,True
1,ASSISTments,skill_catalog.parquet,161,16637,True,True
2,ASSISTments,problem_catalog.parquet,39779,978772,True,True
3,ASSISTments,early_skill_history.parquet,260768,8707997,True,True
4,ASSISTments,future_skill_relevance.parquet,174823,4960293,True,True
5,ASSISTments,early_problem_history.parquet,4052498,76911901,True,True
6,ASSISTments,future_problem_relevance.parquet,1783782,35311927,True,True
7,ASSISTments,problem_skill_map.parquet,17826,176578,True,True
8,ASSISTments,skill_name_id_map.parquet,194,10524,True,True
9,KDD,learner_splits.parquet,565,46435,True,True


Validated 18 Phase 2 Parquet artifacts without loading the large histories.


## Step 3 — Define datasets, tasks and relevance labels

Later attempted items are the primary offline target. Later successful items are a sensitivity target. Neither target is interpreted as causal learning benefit.

In [4]:
RELEVANCE_DEFINITIONS = {
    'attempted': 'relevance_binary',
    'successful': 'successful_future_item',
}
TASK_DEFINITIONS = [
    {
        'Dataset': 'ASSISTments', 'Task': 'problem', 'Priority': 'primary',
        'CatalogFile': 'problem_catalog.parquet',
        'EarlyHistoryFile': 'early_problem_history.parquet',
        'FutureRelevanceFile': 'future_problem_relevance.parquet',
        'EvaluableColumn': 'evaluable_problem',
        'ColdStartColumn': 'cold_start_problem_history',
        'ClusterUse': 'accepted_ablation',
    },
    {
        'Dataset': 'ASSISTments', 'Task': 'skill', 'Priority': 'secondary',
        'CatalogFile': 'skill_catalog.parquet',
        'EarlyHistoryFile': 'early_skill_history.parquet',
        'FutureRelevanceFile': 'future_skill_relevance.parquet',
        'EvaluableColumn': 'evaluable_skill',
        'ColdStartColumn': 'cold_start_skill_history',
        'ClusterUse': 'accepted_ablation',
    },
    {
        'Dataset': 'KDD', 'Task': 'problem', 'Priority': 'external_validation',
        'CatalogFile': 'problem_catalog.parquet',
        'EarlyHistoryFile': 'early_problem_history.parquet',
        'FutureRelevanceFile': 'future_problem_relevance.parquet',
        'EvaluableColumn': 'evaluable_problem',
        'ColdStartColumn': 'cold_start_problem_history',
        'ClusterUse': 'exploratory_ablation_only',
    },
    {
        'Dataset': 'KDD', 'Task': 'skill', 'Priority': 'external_validation',
        'CatalogFile': 'skill_catalog.parquet',
        'EarlyHistoryFile': 'early_skill_history.parquet',
        'FutureRelevanceFile': 'future_skill_relevance.parquet',
        'EvaluableColumn': 'evaluable_skill',
        'ColdStartColumn': 'cold_start_skill_history',
        'ClusterUse': 'exploratory_ablation_only',
    },
]
task_table = pd.DataFrame(TASK_DEFINITIONS)
summary_lookup = benchmark_summary.set_index('Dataset')
task_table['CandidateCount'] = task_table.apply(
    lambda row: int(summary_lookup.loc[row['Dataset'], 'ProblemCandidates' if row['Task'] == 'problem' else 'SkillCandidates']),
    axis=1,
)
task_table['ValidationEvaluableRate'] = task_table.apply(
    lambda row: float(summary_lookup.loc[row['Dataset'], 'ValidationProblemEvaluableRate' if row['Task'] == 'problem' else 'ValidationSkillEvaluableRate']),
    axis=1,
)
task_table['ValidationColdStartRate'] = task_table.apply(
    lambda row: float(summary_lookup.loc[row['Dataset'], 'ValidationProblemColdStartRate' if row['Task'] == 'problem' else 'ValidationSkillColdStartRate']),
    axis=1,
)
assert len(task_table) == 4
assert task_table['CandidateCount'].gt(0).all()
assert set(RELEVANCE_DEFINITIONS) == {'attempted', 'successful'}
display(task_table)

,Dataset,Task,Priority,CatalogFile,EarlyHistoryFile,FutureRelevanceFile,EvaluableColumn,ColdStartColumn,ClusterUse,CandidateCount,ValidationEvaluableRate,ValidationColdStartRate
0,ASSISTments,problem,primary,problem_catalog.parquet,early_problem_history.parquet,future_problem_relevance.parquet,evaluable_problem,cold_start_problem_history,accepted_ablation,39779,0.899355,0.024599
1,ASSISTments,skill,secondary,skill_catalog.parquet,early_skill_history.parquet,future_skill_relevance.parquet,evaluable_skill,cold_start_skill_history,accepted_ablation,161,0.588271,0.375731
2,KDD,problem,external_validation,problem_catalog.parquet,early_problem_history.parquet,future_problem_relevance.parquet,evaluable_problem,cold_start_problem_history,exploratory_ablation_only,855,0.982301,0.000000
3,KDD,skill,external_validation,skill_catalog.parquet,early_skill_history.parquet,future_skill_relevance.parquet,evaluable_skill,cold_start_skill_history,exploratory_ablation_only,99,0.991150,0.000000


## Step 4 — Candidate policies

Candidate-policy filtering operates on already-generated sparse score rows. It never creates a dense learner-by-catalogue cross-product.

In [5]:
CANDIDATE_POLICIES = {
    'all_supported': {
        'exclude_seen': False,
        'description': 'Allow supported items seen previously; repeated practice is valid.',
    },
    'novel_only': {
        'exclude_seen': True,
        'description': 'Exclude items present in the learner early history.',
    },
}

def apply_candidate_policy(scored_candidates, early_history, policy):
    if policy not in CANDIDATE_POLICIES:
        raise ValueError(f'Unknown candidate policy: {policy}')
    required = {'learner_id', 'item_id', 'score'}
    if not required.issubset(scored_candidates.columns):
        raise ValueError(f'Scored candidates require columns: {sorted(required)}')
    candidates = scored_candidates.copy()
    if not CANDIDATE_POLICIES[policy]['exclude_seen']:
        return candidates
    seen = (
        early_history.loc[early_history['in_candidate_catalog'], ['learner_id', 'item_id']]
        .drop_duplicates().assign(_seen=True)
    )
    candidates = candidates.merge(seen, on=['learner_id', 'item_id'], how='left')
    return candidates[candidates['_seen'].ne(True)].drop(columns='_seen')

def assert_candidate_subset(scored_candidates, catalog):
    candidate_items = set(scored_candidates['item_id'])
    catalog_items = set(catalog['item_id'])
    unknown = candidate_items - catalog_items
    assert not unknown, f'{len(unknown)} scored items are outside the candidate catalogue.'


## Step 5 — Ranking, coverage and cold-start metrics

Only learners with at least one in-catalog relevant future item are evaluable. Incomplete recommendation lists are penalised using K as the Precision@K denominator.

In [6]:
def metrics_for_user(recommended_items, relevant_items, k):
    recommended = list(recommended_items[:k])
    relevant = set(relevant_items)
    if not relevant:
        return None
    hits = np.array([item in relevant for item in recommended], dtype=float)
    padded_hits = np.pad(hits, (0, max(0, k - len(hits))))[:k]
    hit_count = padded_hits.sum()
    precision = hit_count / k
    recall = hit_count / len(relevant)
    discounts = np.log2(np.arange(2, k + 2))
    dcg = np.sum(padded_hits / discounts)
    ideal_length = min(len(relevant), k)
    idcg = np.sum(np.ones(ideal_length) / discounts[:ideal_length])
    ndcg = dcg / idcg if idcg else 0.0
    hit_positions = np.flatnonzero(padded_hits)
    average_precision = (
        sum(padded_hits[:position + 1].sum() / (position + 1) for position in hit_positions)
        / min(len(relevant), k)
    )
    return {
        'Precision': precision, 'Recall': recall, 'NDCG': ndcg,
        'MAP': average_precision, 'HitRate': float(hit_count > 0),
    }

def evaluate_recommendations(
    recommendations, relevance, catalog, learner_splits,
    relevance_column, evaluable_column, cold_start_column,
    dataset, task, candidate_policy, relevance_name, ks=METRIC_KS,
    evaluation_learner_ids=None,
):
    started_at = time.perf_counter()
    required_recommendation_columns = {'learner_id', 'item_id', 'score'}
    if not required_recommendation_columns.issubset(recommendations.columns):
        raise ValueError('Recommendations require learner_id, item_id and score.')
    if candidate_policy not in CANDIDATE_POLICIES:
        raise ValueError(f'Unknown candidate policy: {candidate_policy}')
    if relevance_column not in relevance.columns:
        raise ValueError(f'Missing relevance column: {relevance_column}')
    assert not recommendations.duplicated(['learner_id', 'item_id']).any()
    assert np.isfinite(pd.to_numeric(recommendations['score'], errors='coerce')).all()
    assert_candidate_subset(recommendations, catalog)

    ranked = recommendations.sort_values(
        ['learner_id', 'score', 'item_id'], ascending=[True, False, True],
        kind='mergesort',
    )
    relevance_mask = (
        relevance['in_candidate_catalog'] & relevance[relevance_column].eq(1)
    )
    if candidate_policy == 'novel_only':
        if 'seen_in_early' not in relevance.columns:
            raise ValueError('novel_only evaluation requires seen_in_early labels.')
        relevance_mask &= ~relevance['seen_in_early'].fillna(False).astype(bool)
    relevant = relevance.loc[
        relevance_mask, ['learner_id', 'item_id']
    ].drop_duplicates()
    relevant_by_user = relevant.groupby('learner_id')['item_id'].agg(set).to_dict()
    ranked_by_user = ranked.groupby('learner_id')['item_id'].agg(list).to_dict()

    if evaluation_learner_ids is None:
        evaluation_learners = learner_splits[
            learner_splits['cohort'].eq('validation')
        ].copy()
        evaluation_scope = 'validation'
    else:
        evaluation_ids = set(pd.Series(evaluation_learner_ids).astype(str))
        evaluation_learners = learner_splits[
            learner_splits['learner_id'].astype(str).isin(evaluation_ids)
        ].copy()
        assert set(evaluation_learners['learner_id'].astype(str)) == evaluation_ids
        evaluation_scope = 'provided_learner_ids'
    eligible_users = set(evaluation_learners.loc[
        evaluation_learners[evaluable_column], 'learner_id'
    ])
    eligible_users &= set(relevant_by_user)
    segments = {
        'all': eligible_users,
        'cold_start': eligible_users & set(evaluation_learners.loc[evaluation_learners[cold_start_column], 'learner_id']),
        'non_cold_start': eligible_users & set(evaluation_learners.loc[~evaluation_learners[cold_start_column], 'learner_id']),
    }

    rows = []
    catalog_size = len(catalog)
    for segment_name, users in segments.items():
        if not users:
            continue
        for k in ks:
            user_rows = []
            recommended_union = set()
            list_lengths = []
            for learner_id in sorted(users):
                recommended = ranked_by_user.get(learner_id, [])[:k]
                values = metrics_for_user(recommended, relevant_by_user[learner_id], k)
                user_rows.append(values)
                recommended_union.update(recommended)
                list_lengths.append(len(recommended))
            user_metrics = pd.DataFrame(user_rows)
            rows.append({
                'Segment': segment_name, 'K': k,
                'PrecisionAtK': user_metrics['Precision'].mean(),
                'RecallAtK': user_metrics['Recall'].mean(),
                'NDCGAtK': user_metrics['NDCG'].mean(),
                'MAPAtK': user_metrics['MAP'].mean(),
                'HitRateAtK': user_metrics['HitRate'].mean(),
                'CatalogCoverageAtK': len(recommended_union) / catalog_size,
                'MeanRecommendations': float(np.mean(list_lengths)),
                'EvaluatedLearners': len(users),
                'EvaluableRate': len(eligible_users) / len(evaluation_learners),
                'EvaluationScope': evaluation_scope,
            })
    result = pd.DataFrame(rows)
    result.insert(0, 'RelevanceDefinition', relevance_name)
    result.insert(0, 'CandidatePolicy', candidate_policy)
    result.insert(0, 'Task', task)
    result.insert(0, 'Dataset', dataset)
    result['RecommendationMemoryBytes'] = int(recommendations.memory_usage(deep=True).sum())
    result['MetricRuntimeSeconds'] = time.perf_counter() - started_at
    return result

## Save and verify the Phase 3 steps 1–5 setup

These are setup artifacts, not recommendation results. Baseline result files are created only when steps 6 onward are implemented.

In [7]:
contract_validation.to_csv(OUTPUT_ROOT / 'phase3_contract_validation.csv', index=False)
task_table.to_csv(OUTPUT_ROOT / 'phase3_task_definitions.csv', index=False)
phase3_setup_config = {
    'implemented_steps': [1, 2, 3, 4, 5],
    'pending_steps': list(range(6, 17)),
    'benchmark_root': str(BENCHMARK_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'metric_ks': list(METRIC_KS),
    'primary_relevance': PRIMARY_RELEVANCE,
    'relevance_definitions': RELEVANCE_DEFINITIONS,
    'candidate_policies': CANDIDATE_POLICIES,
    'primary_task': 'ASSISTments problem recommendation',
    'dense_cross_product_forbidden': True,
    'validation_future_role': 'evaluation_only',
    'random_state': RANDOM_STATE,
}
with open(OUTPUT_ROOT / 'phase3_setup_config.json', 'w', encoding='utf-8') as file:
    json.dump(phase3_setup_config, file, indent=2)

setup_files = [
    OUTPUT_ROOT / 'phase3_contract_validation.csv',
    OUTPUT_ROOT / 'phase3_task_definitions.csv',
    OUTPUT_ROOT / 'phase3_setup_config.json',
]
setup_manifest = pd.DataFrame([
    {'File': path.name, 'Bytes': path.stat().st_size}
    for path in setup_files
])
setup_manifest.to_csv(OUTPUT_ROOT / 'phase3_setup_manifest.csv', index=False)
assert len(pd.read_csv(OUTPUT_ROOT / 'phase3_contract_validation.csv')) == 18
assert len(pd.read_csv(OUTPUT_ROOT / 'phase3_task_definitions.csv')) == 4
assert len(pd.read_csv(OUTPUT_ROOT / 'phase3_setup_manifest.csv')) == 3
display(setup_manifest)

,File,Bytes
0,phase3_contract_validation.csv,1084
1,phase3_task_definitions.csv,1028
2,phase3_setup_config.json,1022


## Step 6 — Popularity baselines

Two deterministic global baselines are compared for every dataset/task/relevance combination:

- `popularity_discovery_early`: discovery-early interaction counts from the frozen candidate catalogue.
- `popularity_discovery_future`: the number of discovery learners with the relevant future item. Validation-future labels are excluded before scores are fitted.

Recommendations are generated directly as sparse top-20 rows. For `novel_only`, the ranked catalogue is scanned until each learner has up to 20 unseen items; no dense learner-by-item matrix is constructed.

In [8]:
MAX_RECOMMENDATIONS = max(METRIC_KS)

def load_task_tables(task_definition):
    directory = DATASET_DIRS[task_definition['Dataset']]
    return {
        'learners': pd.read_parquet(directory / 'learner_splits.parquet'),
        'catalog': pd.read_parquet(directory / task_definition['CatalogFile']),
        'early_history': pd.read_parquet(directory / task_definition['EarlyHistoryFile']),
        'future_relevance': pd.read_parquet(directory / task_definition['FutureRelevanceFile']),
    }

def top_k_sparse(scored_candidates, max_k=MAX_RECOMMENDATIONS):
    required = {'learner_id', 'item_id', 'score'}
    if not required.issubset(scored_candidates.columns):
        raise ValueError(f'Scored candidates require columns: {sorted(required)}')
    if scored_candidates.empty:
        return pd.DataFrame(columns=['learner_id', 'item_id', 'score', 'rank'])
    assert not scored_candidates.duplicated(['learner_id', 'item_id']).any()
    ranked = scored_candidates[['learner_id', 'item_id', 'score']].sort_values(
        ['learner_id', 'score', 'item_id'],
        ascending=[True, False, True],
        kind='mergesort',
    )
    ranked = ranked.groupby('learner_id', sort=False).head(max_k).copy()
    ranked['rank'] = ranked.groupby('learner_id', sort=False).cumcount() + 1
    return ranked

def global_top_k_recommendations(
    validation_learners, item_scores, early_history, candidate_policy,
    max_k=MAX_RECOMMENDATIONS,
):
    if candidate_policy not in CANDIDATE_POLICIES:
        raise ValueError(f'Unknown candidate policy: {candidate_policy}')
    required = {'item_id', 'score'}
    if not required.issubset(item_scores.columns):
        raise ValueError('Item scores require item_id and score.')
    assert not item_scores['item_id'].duplicated().any()
    assert np.isfinite(pd.to_numeric(item_scores['score'], errors='coerce')).all()
    ranking = list(
        item_scores[['item_id', 'score']]
        .sort_values(['score', 'item_id'], ascending=[False, True], kind='mergesort')
        .itertuples(index=False, name=None)
    )
    learner_ids = sorted(pd.Series(validation_learners).astype(str).unique())
    seen_by_learner = {}
    if candidate_policy == 'novel_only':
        validation_set = set(learner_ids)
        seen = early_history[
            early_history['in_candidate_catalog']
            & early_history['learner_id'].astype(str).isin(validation_set)
        ][['learner_id', 'item_id']].drop_duplicates()
        seen_by_learner = seen.groupby('learner_id')['item_id'].agg(set).to_dict()

    rows = []
    for learner_id in learner_ids:
        excluded = seen_by_learner.get(learner_id, set())
        rank = 0
        for item_id, score in ranking:
            if item_id in excluded:
                continue
            rank += 1
            rows.append((learner_id, item_id, float(score), rank))
            if rank == max_k:
                break
    recommendations = pd.DataFrame(
        rows, columns=['learner_id', 'item_id', 'score', 'rank']
    )
    assert not recommendations.duplicated(['learner_id', 'item_id']).any()
    assert recommendations.groupby('learner_id').size().le(max_k).all()
    return recommendations

def discovery_early_popularity_scores(catalog):
    scores = catalog[
        ['item_id', 'training_interactions', 'training_learners', 'training_popularity']
    ].copy()
    scores['score'] = scores['training_interactions'].astype(float)
    scores['normalised_score'] = scores['training_popularity'].astype(float)
    scores['score_source'] = 'discovery_early_interactions'
    return scores

def discovery_future_popularity_scores(
    relevance, learners, catalog, relevance_column,
):
    discovery_ids = set(
        learners.loc[learners['cohort'].eq('discovery'), 'learner_id'].astype(str)
    )
    validation_ids = set(
        learners.loc[learners['cohort'].eq('validation'), 'learner_id'].astype(str)
    )
    training_labels = relevance[
        relevance['learner_id'].astype(str).isin(discovery_ids)
        & relevance['in_candidate_catalog']
    ][['learner_id', 'item_id', relevance_column]].copy()
    assert set(training_labels['learner_id']).issubset(discovery_ids)
    assert set(training_labels['learner_id']).isdisjoint(validation_ids)
    assert not training_labels.duplicated(['learner_id', 'item_id']).any()
    relevant_labels = training_labels[training_labels[relevance_column].eq(1)]
    counts = (
        relevant_labels.groupby('item_id')['learner_id']
        .nunique().rename('discovery_relevant_learners').reset_index()
    )
    scores = catalog[['item_id']].merge(counts, on='item_id', how='left')
    scores['discovery_relevant_learners'] = (
        scores['discovery_relevant_learners'].fillna(0).astype('int64')
    )
    scores['score'] = scores['discovery_relevant_learners'].astype(float)
    total = scores['score'].sum()
    scores['normalised_score'] = scores['score'] / total if total else 0.0
    scores['score_source'] = f'discovery_future_{relevance_column}'
    return scores

In [9]:
popularity_metric_parts = []
popularity_recommendation_parts = []
popularity_score_parts = []

for task_definition in TASK_DEFINITIONS:
    dataset = task_definition['Dataset']
    task = task_definition['Task']
    tables = load_task_tables(task_definition)
    learners = tables['learners']
    catalog = tables['catalog']
    early_history = tables['early_history']
    future_relevance = tables['future_relevance']
    validation_ids = learners.loc[
        learners['cohort'].eq('validation'), 'learner_id'
    ].astype(str)

    early_scores = discovery_early_popularity_scores(catalog)
    assert_candidate_subset(early_scores[['item_id', 'score']], catalog)

    for relevance_name, relevance_column in RELEVANCE_DEFINITIONS.items():
        future_scores = discovery_future_popularity_scores(
            future_relevance, learners, catalog, relevance_column
        )
        models = {
            'popularity_discovery_early': early_scores,
            'popularity_discovery_future': future_scores,
        }
        for model_name, item_scores in models.items():
            score_output = item_scores.copy()
            score_output.insert(0, 'RelevanceDefinition', relevance_name)
            score_output.insert(0, 'Model', model_name)
            score_output.insert(0, 'Task', task)
            score_output.insert(0, 'Dataset', dataset)
            popularity_score_parts.append(score_output)

            for candidate_policy in CANDIDATE_POLICIES:
                recommendation_started = time.perf_counter()
                recommendations = global_top_k_recommendations(
                    validation_ids, item_scores, early_history,
                    candidate_policy, MAX_RECOMMENDATIONS,
                )
                assert_candidate_subset(recommendations, catalog)
                recommendation_runtime = time.perf_counter() - recommendation_started
                metrics = evaluate_recommendations(
                    recommendations, future_relevance, catalog, learners,
                    relevance_column, task_definition['EvaluableColumn'],
                    task_definition['ColdStartColumn'], dataset, task,
                    candidate_policy, relevance_name,
                )
                metrics.insert(4, 'Model', model_name)
                metrics['RecommendationBuildRuntimeSeconds'] = recommendation_runtime
                popularity_metric_parts.append(metrics)

                recommendation_output = recommendations.copy()
                recommendation_output.insert(0, 'RelevanceDefinition', relevance_name)
                recommendation_output.insert(0, 'CandidatePolicy', candidate_policy)
                recommendation_output.insert(0, 'Model', model_name)
                recommendation_output.insert(0, 'Task', task)
                recommendation_output.insert(0, 'Dataset', dataset)
                popularity_recommendation_parts.append(recommendation_output)

popularity_metrics = pd.concat(popularity_metric_parts, ignore_index=True)
popularity_recommendations = pd.concat(
    popularity_recommendation_parts, ignore_index=True
)
popularity_scores = pd.concat(popularity_score_parts, ignore_index=True)
assert set(popularity_metrics['Model']) == {
    'popularity_discovery_early', 'popularity_discovery_future'
}
assert set(popularity_metrics['CandidatePolicy']) == set(CANDIDATE_POLICIES)
assert set(popularity_metrics['RelevanceDefinition']) == set(RELEVANCE_DEFINITIONS)
display(popularity_metrics.sort_values(
    ['Dataset', 'Task', 'RelevanceDefinition', 'CandidatePolicy', 'Model', 'Segment', 'K']
))

,Dataset,Task,CandidatePolicy,RelevanceDefinition,Model,Segment,K,PrecisionAtK,RecallAtK,NDCGAtK,MAPAtK,HitRateAtK,CatalogCoverageAtK,MeanRecommendations,EvaluatedLearners,EvaluableRate,EvaluationScope,RecommendationMemoryBytes,MetricRuntimeSeconds,RecommendationBuildRuntimeSeconds
0,ASSISTments,problem,all_supported,attempted,popularity_discovery_early,all,5,0.011341,0.003534,0.011477,0.009014,0.021014,0.000126,5.000000,5996,0.899355,validation,17866552,5.884369,0.283054
1,ASSISTments,problem,all_supported,attempted,popularity_discovery_early,all,10,0.011474,0.007234,0.011900,0.008486,0.027352,0.000251,10.000000,5996,0.899355,validation,17866552,5.884369,0.283054
2,ASSISTments,problem,all_supported,attempted,popularity_discovery_early,all,20,0.011116,0.013276,0.014485,0.009746,0.032188,0.000503,20.000000,5996,0.899355,validation,17866552,5.884369,0.283054
3,ASSISTments,problem,all_supported,attempted,popularity_discovery_early,cold_start,5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000126,5.000000,21,0.899355,validation,17866552,5.884369,0.283054
4,ASSISTments,problem,all_supported,attempted,popularity_discovery_early,cold_start,10,0.000000,0.000000,0.000000,0.000000,0.000000,0.000251,10.000000,21,0.899355,validation,17866552,5.884369,0.283054
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,KDD,skill,novel_only,successful,popularity_discovery_future,all,10,0.294318,0.597980,0.509490,0.402189,0.818182,0.808081,9.965909,88,0.778761,validation,462925,0.062594,0.013502
236,KDD,skill,novel_only,successful,popularity_discovery_future,all,20,0.204545,0.783098,0.566299,0.434034,0.931818,0.929293,19.715909,88,0.778761,validation,462925,0.062594,0.013502
237,KDD,skill,novel_only,successful,popularity_discovery_future,non_cold_start,5,0.377273,0.382336,0.452428,0.384201,0.715909,0.585859,5.000000,88,0.778761,validation,462925,0.062594,0.013502
238,KDD,skill,novel_only,successful,popularity_discovery_future,non_cold_start,10,0.294318,0.597980,0.509490,0.402189,0.818182,0.808081,9.965909,88,0.778761,validation,462925,0.062594,0.013502


## Step 7 — Weak-skill baseline

This baseline applies only to skill recommendation. It scores validation learners' supported `weak` and `developing` skills as `(1 - empirical_bayes_mastery) * mastery_evidence_confidence`.

`insufficient_evidence` remains a separate diagnostic state and is never interpreted as demonstrated weakness. The model has no unobserved-skill transfer signal, so `novel_only` correctly produces empty recommendation lists instead of adding an undeclared popularity backfill.

In [10]:
WEAK_SKILL_STATES = {'weak', 'developing'}
weak_skill_metric_parts = []
weak_skill_recommendation_parts = []
weak_skill_score_parts = []
weak_skill_diagnostic_parts = []

for task_definition in [row for row in TASK_DEFINITIONS if row['Task'] == 'skill']:
    dataset = task_definition['Dataset']
    tables = load_task_tables(task_definition)
    learners = tables['learners']
    catalog = tables['catalog']
    early_history = tables['early_history']
    future_relevance = tables['future_relevance']
    validation_ids = set(
        learners.loc[learners['cohort'].eq('validation'), 'learner_id'].astype(str)
    )
    validation_history = early_history[
        early_history['learner_id'].astype(str).isin(validation_ids)
        & early_history['in_candidate_catalog']
    ].copy()

    diagnostics = (
        validation_history.groupby('skill_state', dropna=False)
        .size().rename('LearnerSkillRows').reset_index()
    )
    diagnostics.insert(0, 'Dataset', dataset)
    diagnostics['UsedAsWeaknessEvidence'] = diagnostics['skill_state'].isin(
        WEAK_SKILL_STATES
    )
    weak_skill_diagnostic_parts.append(diagnostics)

    scored = validation_history[
        validation_history['skill_state'].isin(WEAK_SKILL_STATES)
    ][
        ['learner_id', 'item_id', 'empirical_bayes_mastery',
         'mastery_evidence_confidence', 'skill_state']
    ].copy()
    assert not scored['skill_state'].eq('insufficient_evidence').any()
    assert scored['empirical_bayes_mastery'].between(0, 1).all()
    assert scored['mastery_evidence_confidence'].between(0, 1).all()
    scored['score'] = (
        (1.0 - scored['empirical_bayes_mastery'])
        * scored['mastery_evidence_confidence']
    )
    assert np.isfinite(scored['score']).all()
    assert_candidate_subset(scored, catalog)

    score_output = scored.copy()
    score_output.insert(0, 'Model', 'weak_skill_mastery_confidence')
    score_output.insert(0, 'Task', 'skill')
    score_output.insert(0, 'Dataset', dataset)
    weak_skill_score_parts.append(score_output)

    for candidate_policy in CANDIDATE_POLICIES:
        recommendation_started = time.perf_counter()
        policy_scores = apply_candidate_policy(
            scored, validation_history, candidate_policy
        )
        recommendations = top_k_sparse(policy_scores, MAX_RECOMMENDATIONS)
        if candidate_policy == 'novel_only':
            assert recommendations.empty
        recommendation_runtime = time.perf_counter() - recommendation_started
        assert_candidate_subset(recommendations, catalog)

        for relevance_name, relevance_column in RELEVANCE_DEFINITIONS.items():
            metrics = evaluate_recommendations(
                recommendations, future_relevance, catalog, learners,
                relevance_column, task_definition['EvaluableColumn'],
                task_definition['ColdStartColumn'], dataset, 'skill',
                candidate_policy, relevance_name,
            )
            metrics.insert(4, 'Model', 'weak_skill_mastery_confidence')
            metrics['RecommendationBuildRuntimeSeconds'] = recommendation_runtime
            metrics['PolicyInterpretation'] = np.where(
                metrics['CandidatePolicy'].eq('novel_only'),
                'no_unseen_transfer_signal', 'personalised_weakness_ranking',
            )
            weak_skill_metric_parts.append(metrics)

            recommendation_output = recommendations.copy()
            recommendation_output.insert(0, 'RelevanceDefinition', relevance_name)
            recommendation_output.insert(0, 'CandidatePolicy', candidate_policy)
            recommendation_output.insert(0, 'Model', 'weak_skill_mastery_confidence')
            recommendation_output.insert(0, 'Task', 'skill')
            recommendation_output.insert(0, 'Dataset', dataset)
            weak_skill_recommendation_parts.append(recommendation_output)

weak_skill_metrics = pd.concat(weak_skill_metric_parts, ignore_index=True)
weak_skill_recommendations = pd.concat(
    [part for part in weak_skill_recommendation_parts if not part.empty],
    ignore_index=True,
)
weak_skill_scores = pd.concat(weak_skill_score_parts, ignore_index=True)
weak_skill_diagnostics = pd.concat(
    weak_skill_diagnostic_parts, ignore_index=True
)
assert not weak_skill_diagnostics.loc[
    weak_skill_diagnostics['skill_state'].eq('insufficient_evidence'),
    'UsedAsWeaknessEvidence',
].any()
display(weak_skill_metrics.sort_values(
    ['Dataset', 'RelevanceDefinition', 'CandidatePolicy', 'Segment', 'K']
))
display(weak_skill_diagnostics.sort_values(['Dataset', 'skill_state']))

,Dataset,Task,CandidatePolicy,RelevanceDefinition,Model,Segment,K,PrecisionAtK,RecallAtK,NDCGAtK,...,HitRateAtK,CatalogCoverageAtK,MeanRecommendations,EvaluatedLearners,EvaluableRate,EvaluationScope,RecommendationMemoryBytes,MetricRuntimeSeconds,RecommendationBuildRuntimeSeconds,PolicyInterpretation
0,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,all,5,0.244773,0.197409,0.321089,...,0.616012,0.844720,3.291688,3922,0.588271,validation,2980604,2.593235,0.026243,personalised_weakness_ranking
1,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,all,10,0.168256,0.227048,0.295214,...,0.636410,0.869565,4.569862,3922,0.588271,validation,2980604,2.593235,0.026243,personalised_weakness_ranking
2,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,all,20,0.097081,0.238813,0.275014,...,0.639215,0.875776,5.279704,3922,0.588271,validation,2980604,2.593235,0.026243,personalised_weakness_ranking
3,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,cold_start,5,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,133,0.588271,validation,2980604,2.593235,0.026243,personalised_weakness_ranking
4,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,cold_start,10,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,133,0.588271,validation,2980604,2.593235,0.026243,personalised_weakness_ranking
5,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,cold_start,20,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,133,0.588271,validation,2980604,2.593235,0.026243,personalised_weakness_ranking
6,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,non_cold_start,5,0.253365,0.204338,0.332360,...,0.637635,0.844720,3.407231,3789,0.588271,validation,2980604,2.593235,0.026243,personalised_weakness_ranking
7,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,non_cold_start,10,0.174162,0.235017,0.305576,...,0.658749,0.869565,4.730272,3789,0.588271,validation,2980604,2.593235,0.026243,personalised_weakness_ranking
8,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,non_cold_start,20,0.100488,0.247196,0.284667,...,0.661652,0.875776,5.465030,3789,0.588271,validation,2980604,2.593235,0.026243,personalised_weakness_ranking
18,ASSISTments,skill,novel_only,attempted,weak_skill_mastery_confidence,all,5,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,3441,0.516124,validation,132,1.537552,0.040059,no_unseen_transfer_signal


,Dataset,skill_state,LearnerSkillRows,UsedAsWeaknessEvidence
0,ASSISTments,developing,14898,True
1,ASSISTments,insufficient_evidence,17882,False
2,ASSISTments,mastered,12039,False
3,ASSISTments,weak,6963,True
4,KDD,developing,1001,True
5,KDD,insufficient_evidence,500,False
6,KDD,mastered,1175,False
7,KDD,weak,1094,True


## Save and verify Steps 6–7 artifacts

These are staged baseline artifacts. The consolidated Phase 3 filenames required by Step 15 will be produced only after all baseline families are implemented.

In [11]:
steps_6_7_metrics = pd.concat(
    [popularity_metrics, weak_skill_metrics], ignore_index=True, sort=False
)
steps_6_7_recommendations = pd.concat(
    [popularity_recommendations, weak_skill_recommendations],
    ignore_index=True, sort=False,
)

steps_6_7_paths = {
    'metrics': OUTPUT_ROOT / 'steps_6_7_metrics.csv',
    'recommendations': OUTPUT_ROOT / 'steps_6_7_recommendations.parquet',
    'popularity_scores': OUTPUT_ROOT / 'step6_popularity_scores.parquet',
    'weak_skill_scores': OUTPUT_ROOT / 'step7_weak_skill_scores.parquet',
    'weak_skill_diagnostics': OUTPUT_ROOT / 'step7_weak_skill_diagnostics.csv',
    'config': OUTPUT_ROOT / 'steps_6_7_config.json',
}
steps_6_7_metrics.to_csv(steps_6_7_paths['metrics'], index=False)
steps_6_7_recommendations.to_parquet(
    steps_6_7_paths['recommendations'], index=False, compression='snappy'
)
popularity_scores.to_parquet(
    steps_6_7_paths['popularity_scores'], index=False, compression='snappy'
)
weak_skill_scores.to_parquet(
    steps_6_7_paths['weak_skill_scores'], index=False, compression='snappy'
)
weak_skill_diagnostics.to_csv(
    steps_6_7_paths['weak_skill_diagnostics'], index=False
)

steps_6_7_config = {
    'implemented_steps': [6, 7],
    'metric_ks': list(METRIC_KS),
    'maximum_saved_recommendations_per_learner': MAX_RECOMMENDATIONS,
    'popularity_models': [
        'popularity_discovery_early', 'popularity_discovery_future'
    ],
    'discovery_future_training_learners_only': True,
    'validation_future_role': 'evaluation_only',
    'candidate_policies': list(CANDIDATE_POLICIES),
    'relevance_definitions': RELEVANCE_DEFINITIONS,
    'weak_skill_model': 'weak_skill_mastery_confidence',
    'weak_skill_formula': '(1 - empirical_bayes_mastery) * mastery_evidence_confidence',
    'weak_skill_states': sorted(WEAK_SKILL_STATES),
    'insufficient_evidence_handling': 'diagnostic_only_not_weakness',
    'weak_skill_novel_only_handling': 'empty_no_unseen_transfer_signal',
    'dense_learner_item_matrix_constructed': False,
}
with open(steps_6_7_paths['config'], 'w', encoding='utf-8') as file:
    json.dump(steps_6_7_config, file, indent=2)

steps_6_7_manifest = pd.DataFrame([
    {
        'Artifact': name,
        'File': path.name,
        'Rows': (
            pq.ParquetFile(path).metadata.num_rows
            if path.suffix == '.parquet'
            else (len(pd.read_csv(path)) if path.suffix == '.csv' else 1)
        ),
        'Bytes': path.stat().st_size,
    }
    for name, path in steps_6_7_paths.items()
])
steps_6_7_manifest.to_csv(
    OUTPUT_ROOT / 'steps_6_7_artifact_manifest.csv', index=False
)
assert len(steps_6_7_manifest) == 6
assert pq.ParquetFile(
    steps_6_7_paths['recommendations']
).metadata.num_rows == len(steps_6_7_recommendations)
assert len(pd.read_csv(steps_6_7_paths['metrics'])) == len(steps_6_7_metrics)
display(steps_6_7_manifest)

,Artifact,File,Rows,Bytes
0,metrics,steps_6_7_metrics.csv,300,82484
1,recommendations,steps_6_7_recommendations.parquet,2215406,1109216
2,popularity_scores,step6_popularity_scores.parquet,163576,1193533
3,weak_skill_scores,step7_weak_skill_scores.parquet,23956,339990
4,weak_skill_diagnostics,step7_weak_skill_diagnostics.csv,8,304
5,config,steps_6_7_config.json,1,909


## Step 8 — Content-based problem recommendation

The content model builds validation-time learner profiles from early weak/developing skill evidence and early problem metadata. Candidate problems come only from the discovery-early problem catalogue and discovery-early problem-to-skill map.

The score combines weak-skill alignment, difficulty suitability, problem-type affinity and hierarchy affinity. Discovery-early popularity is used only as a deterministic numerical tie-breaker, not as a substantive model component.

In [12]:
CONTENT_WEIGHTS = {
    'weak_skill_alignment': 0.65,
    'difficulty_suitability': 0.20,
    'problem_type_affinity': 0.10,
    'hierarchy_affinity': 0.05,
}
POPULARITY_TIE_BREAK_WEIGHT = 1e-9

def build_content_problem_scores(task_definition):
    dataset = task_definition['Dataset']
    directory = DATASET_DIRS[dataset]
    learners = pd.read_parquet(directory / 'learner_splits.parquet')
    skill_history = pd.read_parquet(directory / 'early_skill_history.parquet')
    problem_history = pd.read_parquet(directory / 'early_problem_history.parquet')
    problem_catalog = pd.read_parquet(directory / 'problem_catalog.parquet')
    problem_skill_map = pd.read_parquet(directory / 'problem_skill_map.parquet')

    assert problem_catalog['catalog_source'].eq('discovery_early_only').all()
    assert problem_skill_map['mapping_source'].eq('discovery_early_only').all()
    validation_ids = set(
        learners.loc[learners['cohort'].eq('validation'), 'learner_id'].astype(str)
    )
    skill_need = skill_history[
        skill_history['learner_id'].astype(str).isin(validation_ids)
        & skill_history['in_candidate_catalog']
        & skill_history['skill_state'].isin(WEAK_SKILL_STATES)
    ][
        ['learner_id', 'item_id', 'empirical_bayes_mastery',
         'mastery_evidence_confidence', 'skill_state']
    ].copy()
    skill_need['weakness'] = (
        (1.0 - skill_need['empirical_bayes_mastery'])
        * skill_need['mastery_evidence_confidence']
    )
    assert np.isfinite(skill_need['weakness']).all()

    mapped = skill_need.merge(
        problem_skill_map[
            ['problem_item_id', 'skill_item_id', 'association_share']
        ],
        left_on='item_id', right_on='skill_item_id', how='inner',
        validate='many_to_many',
    )
    mapped['alignment_contribution'] = (
        mapped['weakness'] * mapped['association_share']
    )
    mapped['mastery_contribution'] = (
        mapped['empirical_bayes_mastery'] * mapped['association_share']
    )
    candidate_scores = mapped.groupby(
        ['learner_id', 'problem_item_id'], sort=False
    ).agg(
        weak_skill_alignment=('alignment_contribution', 'sum'),
        weighted_mastery=('mastery_contribution', 'sum'),
        mapped_association_coverage=('association_share', 'sum'),
        matched_weak_skills=('skill_item_id', 'nunique'),
    ).reset_index().rename(columns={'problem_item_id': 'item_id'})
    candidate_scores['profile_mastery'] = (
        candidate_scores['weighted_mastery']
        / candidate_scores['mapped_association_coverage'].clip(lower=1e-12)
    ).clip(0, 1)

    catalog_metadata = problem_catalog[
        ['item_id', 'difficulty_proxy', 'problem_type', 'hierarchy',
         'training_popularity']
    ].copy()
    catalog_metadata['difficulty_proxy'] = (
        catalog_metadata['difficulty_proxy'].fillna(0.5).clip(0, 1)
    )
    catalog_metadata['problem_type'] = (
        catalog_metadata['problem_type'].fillna('__missing__').astype(str)
    )
    catalog_metadata['hierarchy'] = (
        catalog_metadata['hierarchy'].fillna('__missing__').astype(str)
    )

    early_problem_profiles = problem_history[
        problem_history['learner_id'].astype(str).isin(validation_ids)
        & problem_history['in_candidate_catalog']
    ][['learner_id', 'item_id', 'early_interaction_count']].merge(
        catalog_metadata[['item_id', 'problem_type', 'hierarchy']],
        on='item_id', how='inner', validate='many_to_one',
    )
    profile_totals = early_problem_profiles.groupby('learner_id')[
        'early_interaction_count'
    ].transform('sum').clip(lower=1)
    early_problem_profiles['interaction_share'] = (
        early_problem_profiles['early_interaction_count'] / profile_totals
    )
    type_affinity = early_problem_profiles.groupby(
        ['learner_id', 'problem_type'], sort=False
    )['interaction_share'].sum().rename('problem_type_affinity').reset_index()
    hierarchy_affinity = early_problem_profiles.groupby(
        ['learner_id', 'hierarchy'], sort=False
    )['interaction_share'].sum().rename('hierarchy_affinity').reset_index()

    candidate_scores = candidate_scores.merge(
        catalog_metadata, on='item_id', how='inner', validate='many_to_one'
    ).merge(
        type_affinity, on=['learner_id', 'problem_type'], how='left',
        validate='many_to_one',
    ).merge(
        hierarchy_affinity, on=['learner_id', 'hierarchy'], how='left',
        validate='many_to_one',
    )
    candidate_scores[['problem_type_affinity', 'hierarchy_affinity']] = (
        candidate_scores[['problem_type_affinity', 'hierarchy_affinity']]
        .fillna(0.0)
    )
    candidate_scores['difficulty_suitability'] = (
        1.0 - (
            candidate_scores['difficulty_proxy']
            - candidate_scores['profile_mastery']
        ).abs()
    ).clip(0, 1)
    candidate_scores['score'] = (
        CONTENT_WEIGHTS['weak_skill_alignment']
        * candidate_scores['weak_skill_alignment'].clip(0, 1)
        + CONTENT_WEIGHTS['difficulty_suitability']
        * candidate_scores['difficulty_suitability']
        + CONTENT_WEIGHTS['problem_type_affinity']
        * candidate_scores['problem_type_affinity']
        + CONTENT_WEIGHTS['hierarchy_affinity']
        * candidate_scores['hierarchy_affinity']
        + POPULARITY_TIE_BREAK_WEIGHT
        * candidate_scores['training_popularity']
    )
    assert np.isfinite(candidate_scores['score']).all()
    assert_candidate_subset(candidate_scores, problem_catalog)
    diagnostics = pd.DataFrame([{
        'Dataset': dataset,
        'ValidationLearners': len(validation_ids),
        'LearnersWithWeakOrDevelopingSkills': skill_need['learner_id'].nunique(),
        'WeakOrDevelopingLearnerSkillRows': len(skill_need),
        'SparseLearnerProblemCandidates': len(candidate_scores),
        'DenseMatrixConstructed': False,
    }])
    return candidate_scores, diagnostics

In [13]:
content_metric_parts = []
content_recommendation_parts = []
content_score_parts = []
content_diagnostic_parts = []

for task_definition in [row for row in TASK_DEFINITIONS if row['Task'] == 'problem']:
    dataset = task_definition['Dataset']
    tables = load_task_tables(task_definition)
    profile_started = time.perf_counter()
    content_scores_for_task, content_diagnostics_for_task = (
        build_content_problem_scores(task_definition)
    )
    profile_runtime = time.perf_counter() - profile_started
    content_scores_for_task.insert(0, 'Model', 'content_problem')
    content_scores_for_task.insert(0, 'Task', 'problem')
    content_scores_for_task.insert(0, 'Dataset', dataset)
    content_score_parts.append(content_scores_for_task)
    content_diagnostics_for_task['ProfileBuildRuntimeSeconds'] = profile_runtime
    content_diagnostic_parts.append(content_diagnostics_for_task)

    model_scores = content_scores_for_task.drop(
        columns=['Dataset', 'Task', 'Model']
    )
    for candidate_policy in CANDIDATE_POLICIES:
        recommendation_started = time.perf_counter()
        policy_scores = apply_candidate_policy(
            model_scores, tables['early_history'], candidate_policy
        )
        recommendations = top_k_sparse(policy_scores, MAX_RECOMMENDATIONS)
        recommendation_runtime = time.perf_counter() - recommendation_started
        assert_candidate_subset(recommendations, tables['catalog'])

        for relevance_name, relevance_column in RELEVANCE_DEFINITIONS.items():
            metrics = evaluate_recommendations(
                recommendations, tables['future_relevance'], tables['catalog'],
                tables['learners'], relevance_column,
                task_definition['EvaluableColumn'],
                task_definition['ColdStartColumn'], dataset, 'problem',
                candidate_policy, relevance_name,
            )
            metrics.insert(4, 'Model', 'content_problem')
            metrics['ProfileBuildRuntimeSeconds'] = profile_runtime
            metrics['RecommendationBuildRuntimeSeconds'] = recommendation_runtime
            content_metric_parts.append(metrics)

            output = recommendations.copy()
            output.insert(0, 'RelevanceDefinition', relevance_name)
            output.insert(0, 'CandidatePolicy', candidate_policy)
            output.insert(0, 'Model', 'content_problem')
            output.insert(0, 'Task', 'problem')
            output.insert(0, 'Dataset', dataset)
            content_recommendation_parts.append(output)

content_metrics = pd.concat(content_metric_parts, ignore_index=True)
content_recommendations = pd.concat(
    content_recommendation_parts, ignore_index=True
)
content_scores = pd.concat(content_score_parts, ignore_index=True)
content_diagnostics = pd.concat(content_diagnostic_parts, ignore_index=True)
display(content_metrics.sort_values(
    ['Dataset', 'RelevanceDefinition', 'CandidatePolicy', 'Segment', 'K']
))
display(content_diagnostics)

,Dataset,Task,CandidatePolicy,RelevanceDefinition,Model,Segment,K,PrecisionAtK,RecallAtK,NDCGAtK,...,HitRateAtK,CatalogCoverageAtK,MeanRecommendations,EvaluatedLearners,EvaluableRate,EvaluationScope,RecommendationMemoryBytes,MetricRuntimeSeconds,ProfileBuildRuntimeSeconds,RecommendationBuildRuntimeSeconds
0,ASSISTments,problem,all_supported,attempted,content_problem,all,5,0.007839,0.001984,0.008151,...,0.029853,0.126499,3.030187,5996,0.899355,validation,10742818,4.220810,27.263493,12.170099
1,ASSISTments,problem,all_supported,attempted,content_problem,all,10,0.007038,0.003517,0.007825,...,0.049366,0.176978,6.059373,5996,0.899355,validation,10742818,4.220810,27.263493,12.170099
2,ASSISTments,problem,all_supported,attempted,content_problem,all,20,0.007221,0.006765,0.008746,...,0.079720,0.228211,12.104403,5996,0.899355,validation,10742818,4.220810,27.263493,12.170099
3,ASSISTments,problem,all_supported,attempted,content_problem,cold_start,5,0.009524,0.047619,0.047619,...,0.047619,0.001031,2.142857,21,0.899355,validation,10742818,4.220810,27.263493,12.170099
4,ASSISTments,problem,all_supported,attempted,content_problem,cold_start,10,0.004762,0.047619,0.047619,...,0.047619,0.001860,4.285714,21,0.899355,validation,10742818,4.220810,27.263493,12.170099
5,ASSISTments,problem,all_supported,attempted,content_problem,cold_start,20,0.002381,0.047619,0.047619,...,0.047619,0.003243,8.571429,21,0.899355,validation,10742818,4.220810,27.263493,12.170099
6,ASSISTments,problem,all_supported,attempted,content_problem,non_cold_start,5,0.007833,0.001824,0.008013,...,0.029791,0.126499,3.033305,5975,0.899355,validation,10742818,4.220810,27.263493,12.170099
7,ASSISTments,problem,all_supported,attempted,content_problem,non_cold_start,10,0.007046,0.003362,0.007685,...,0.049372,0.176877,6.065607,5975,0.899355,validation,10742818,4.220810,27.263493,12.170099
8,ASSISTments,problem,all_supported,attempted,content_problem,non_cold_start,20,0.007238,0.006621,0.008610,...,0.079833,0.228110,12.116820,5975,0.899355,validation,10742818,4.220810,27.263493,12.170099
18,ASSISTments,problem,novel_only,attempted,content_problem,all,5,0.008496,0.002471,0.008895,...,0.032908,0.127555,3.040463,5956,0.893355,validation,10729262,3.598284,27.263493,22.161545


,Dataset,ValidationLearners,LearnersWithWeakOrDevelopingSkills,WeakOrDevelopingLearnerSkillRows,SparseLearnerProblemCandidates,DenseMatrixConstructed,ProfileBuildRuntimeSeconds
0,ASSISTments,6667,3803,21861,6702902,False,27.263493
1,KDD,113,113,2095,67380,False,0.389471


## Step 9 — Learner-neighbour collaborative filtering

Cosine nearest neighbours are fitted on sparse discovery-early learner-item histories. Candidate counts 20, 50 and 100 are selected using an internal discovery-train/discovery-tuning split, attempted relevance, all-supported candidates and NDCG@10.

After selection, the model is refitted on every discovery learner. Validation recommendations are weighted aggregates of discovery neighbours' future relevance; there is no dense learner-item score matrix and no popularity backfill.

In [14]:
from scipy.sparse import csr_matrix, diags
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

NEIGHBOR_CANDIDATES = (20, 50, 100)
DISCOVERY_TUNING_FRACTION = 0.20
NEIGHBOR_QUERY_BATCH_SIZE = 512

def build_sparse_history_matrix(
    history, learner_order, item_order, value_column='early_interaction_count',
):
    learner_order = [str(value) for value in learner_order]
    item_order = [str(value) for value in item_order]
    learner_index = {value: index for index, value in enumerate(learner_order)}
    item_index = {value: index for index, value in enumerate(item_order)}
    subset = history[
        history['learner_id'].astype(str).isin(learner_index)
        & history['item_id'].astype(str).isin(item_index)
        & history['in_candidate_catalog']
    ][['learner_id', 'item_id', value_column]].copy()
    rows = subset['learner_id'].astype(str).map(learner_index).to_numpy()
    columns = subset['item_id'].astype(str).map(item_index).to_numpy()
    values = np.log1p(
        pd.to_numeric(subset[value_column], errors='coerce').fillna(0).to_numpy(float)
    )
    matrix = csr_matrix(
        (values, (rows, columns)),
        shape=(len(learner_order), len(item_order)),
        dtype=np.float32,
    )
    matrix.eliminate_zeros()
    return matrix

def build_discovery_target_matrix(
    relevance, training_learner_order, item_order, relevance_column,
):
    training_learner_order = [str(value) for value in training_learner_order]
    item_order = [str(value) for value in item_order]
    learner_index = {
        value: index for index, value in enumerate(training_learner_order)
    }
    item_index = {value: index for index, value in enumerate(item_order)}
    labels = relevance[
        relevance['learner_id'].astype(str).isin(learner_index)
        & relevance['item_id'].astype(str).isin(item_index)
        & relevance['in_candidate_catalog']
        & relevance[relevance_column].eq(1)
    ][['learner_id', 'item_id']].drop_duplicates()
    assert set(labels['learner_id'].astype(str)).issubset(learner_index)
    rows = labels['learner_id'].astype(str).map(learner_index).to_numpy()
    columns = labels['item_id'].astype(str).map(item_index).to_numpy()
    matrix = csr_matrix(
        (np.ones(len(labels), dtype=np.float32), (rows, columns)),
        shape=(len(training_learner_order), len(item_order)),
        dtype=np.float32,
    )
    return matrix

def query_sparse_neighbors(training_matrix, query_matrix, maximum_neighbors):
    if training_matrix.shape[0] == 0:
        raise ValueError('Cannot fit neighbours without discovery learners.')
    usable_neighbors = min(maximum_neighbors, training_matrix.shape[0])
    model = NearestNeighbors(
        n_neighbors=usable_neighbors, metric='cosine',
        algorithm='brute', n_jobs=-1,
    )
    model.fit(training_matrix)
    distances, indices = model.kneighbors(query_matrix)
    similarities = np.clip(1.0 - distances, 0.0, 1.0)
    return indices, similarities

def aggregate_neighbor_recommendations(
    query_learner_order, training_target_matrix, neighbor_indices,
    neighbor_similarities, item_order, query_history, candidate_policy,
    max_k=MAX_RECOMMENDATIONS, batch_size=NEIGHBOR_QUERY_BATCH_SIZE,
):
    if candidate_policy not in CANDIDATE_POLICIES:
        raise ValueError(f'Unknown candidate policy: {candidate_policy}')
    query_learner_order = [str(value) for value in query_learner_order]
    item_order = [str(value) for value in item_order]
    seen_by_learner = {}
    if candidate_policy == 'novel_only':
        query_set = set(query_learner_order)
        seen = query_history[
            query_history['learner_id'].astype(str).isin(query_set)
            & query_history['in_candidate_catalog']
        ][['learner_id', 'item_id']].drop_duplicates()
        seen_by_learner = seen.groupby('learner_id')['item_id'].agg(set).to_dict()

    rows = []
    training_count = training_target_matrix.shape[0]
    for batch_start in range(0, len(query_learner_order), batch_size):
        batch_end = min(batch_start + batch_size, len(query_learner_order))
        batch_indices = neighbor_indices[batch_start:batch_end]
        batch_similarities = neighbor_similarities[batch_start:batch_end]
        local_rows = np.repeat(
            np.arange(batch_end - batch_start), batch_indices.shape[1]
        )
        weights = csr_matrix(
            (batch_similarities.ravel(), (local_rows, batch_indices.ravel())),
            shape=(batch_end - batch_start, training_count),
            dtype=np.float32,
        )
        weights.eliminate_zeros()
        weight_totals = np.asarray(weights.sum(axis=1)).ravel()
        inverse_totals = np.divide(
            1.0, weight_totals,
            out=np.zeros_like(weight_totals), where=weight_totals > 0,
        )
        weighted_scores = diags(inverse_totals) @ weights @ training_target_matrix
        weighted_scores = weighted_scores.tocsr()
        weighted_scores.eliminate_zeros()

        for local_index, learner_id in enumerate(
            query_learner_order[batch_start:batch_end]
        ):
            start = weighted_scores.indptr[local_index]
            end = weighted_scores.indptr[local_index + 1]
            item_indices = weighted_scores.indices[start:end]
            values = weighted_scores.data[start:end]
            excluded = seen_by_learner.get(learner_id, set())
            candidates = [
                (float(value), item_order[item_index])
                for item_index, value in zip(item_indices, values)
                if value > 0 and item_order[item_index] not in excluded
            ]
            candidates.sort(key=lambda value: (-value[0], value[1]))
            for rank, (score, item_id) in enumerate(
                candidates[:max_k], start=1
            ):
                rows.append((learner_id, item_id, score, rank))
    recommendations = pd.DataFrame(
        rows, columns=['learner_id', 'item_id', 'score', 'rank']
    )
    assert not recommendations.duplicated(['learner_id', 'item_id']).any()
    return recommendations

In [15]:
neighbor_tuning_parts = []
neighbor_selection_rows = []
neighbor_metric_parts = []
neighbor_recommendation_parts = []

for task_definition in TASK_DEFINITIONS:
    dataset = task_definition['Dataset']
    task = task_definition['Task']
    tables = load_task_tables(task_definition)
    learners = tables['learners']
    catalog = tables['catalog']
    early_history = tables['early_history']
    future_relevance = tables['future_relevance']
    item_order = sorted(catalog['item_id'].astype(str).unique())
    discovery_ids = sorted(
        learners.loc[learners['cohort'].eq('discovery'), 'learner_id'].astype(str)
    )
    validation_ids = sorted(
        learners.loc[learners['cohort'].eq('validation'), 'learner_id'].astype(str)
    )
    discovery_train_ids, discovery_tuning_ids = train_test_split(
        discovery_ids, test_size=DISCOVERY_TUNING_FRACTION,
        random_state=RANDOM_STATE,
    )
    discovery_train_ids = sorted(discovery_train_ids)
    discovery_tuning_ids = sorted(discovery_tuning_ids)
    assert set(discovery_train_ids).isdisjoint(discovery_tuning_ids)
    assert set(discovery_train_ids).isdisjoint(validation_ids)
    assert set(discovery_tuning_ids).isdisjoint(validation_ids)

    tuning_training_matrix = build_sparse_history_matrix(
        early_history, discovery_train_ids, item_order
    )
    tuning_query_matrix = build_sparse_history_matrix(
        early_history, discovery_tuning_ids, item_order
    )
    tuning_neighbor_indices, tuning_neighbor_similarities = query_sparse_neighbors(
        tuning_training_matrix, tuning_query_matrix, max(NEIGHBOR_CANDIDATES)
    )
    tuning_target_matrix = build_discovery_target_matrix(
        future_relevance, discovery_train_ids, item_order,
        RELEVANCE_DEFINITIONS['attempted'],
    )
    task_tuning_parts = []
    for neighbor_count in NEIGHBOR_CANDIDATES:
        usable_neighbor_count = min(
            neighbor_count, tuning_neighbor_indices.shape[1]
        )
        tuning_recommendations = aggregate_neighbor_recommendations(
            discovery_tuning_ids, tuning_target_matrix,
            tuning_neighbor_indices[:, :usable_neighbor_count],
            tuning_neighbor_similarities[:, :usable_neighbor_count],
            item_order, early_history, 'all_supported',
        )
        tuning_metrics = evaluate_recommendations(
            tuning_recommendations, future_relevance, catalog, learners,
            RELEVANCE_DEFINITIONS['attempted'],
            task_definition['EvaluableColumn'], task_definition['ColdStartColumn'],
            dataset, task, 'all_supported', 'attempted', ks=(10,),
            evaluation_learner_ids=discovery_tuning_ids,
        )
        tuning_metrics.insert(4, 'Model', 'learner_neighbor_cf_tuning')
        tuning_metrics['NeighborCount'] = neighbor_count
        tuning_metrics['ActualNeighborCount'] = usable_neighbor_count
        tuning_metrics['TuningSource'] = 'discovery_only'
        task_tuning_parts.append(tuning_metrics)
        neighbor_tuning_parts.append(tuning_metrics)

    task_tuning = pd.concat(task_tuning_parts, ignore_index=True)
    selection_candidates = task_tuning[
        task_tuning['Segment'].eq('all') & task_tuning['K'].eq(10)
    ].sort_values(
        ['NDCGAtK', 'NeighborCount'], ascending=[False, True],
        kind='mergesort',
    )
    selected_neighbors = int(selection_candidates.iloc[0]['NeighborCount'])
    neighbor_selection_rows.append({
        'Dataset': dataset,
        'Task': task,
        'SelectedNeighbors': selected_neighbors,
        'SelectionMetric': 'NDCGAtK',
        'SelectionK': 10,
        'SelectionRelevance': 'attempted',
        'SelectionCandidatePolicy': 'all_supported',
        'DiscoveryTrainingLearners': len(discovery_train_ids),
        'DiscoveryTuningLearners': len(discovery_tuning_ids),
        'ValidationLearnersUntouched': len(validation_ids),
    })

    final_training_matrix = build_sparse_history_matrix(
        early_history, discovery_ids, item_order
    )
    validation_query_matrix = build_sparse_history_matrix(
        early_history, validation_ids, item_order
    )
    neighbor_started = time.perf_counter()
    final_neighbor_indices, final_neighbor_similarities = query_sparse_neighbors(
        final_training_matrix, validation_query_matrix, selected_neighbors
    )
    neighbor_query_runtime = time.perf_counter() - neighbor_started
    assert final_neighbor_indices.max() < len(discovery_ids)

    for relevance_name, relevance_column in RELEVANCE_DEFINITIONS.items():
        final_target_matrix = build_discovery_target_matrix(
            future_relevance, discovery_ids, item_order, relevance_column
        )
        for candidate_policy in CANDIDATE_POLICIES:
            recommendation_started = time.perf_counter()
            recommendations = aggregate_neighbor_recommendations(
                validation_ids, final_target_matrix, final_neighbor_indices,
                final_neighbor_similarities, item_order, early_history,
                candidate_policy,
            )
            recommendation_runtime = time.perf_counter() - recommendation_started
            assert_candidate_subset(recommendations, catalog)
            metrics = evaluate_recommendations(
                recommendations, future_relevance, catalog, learners,
                relevance_column, task_definition['EvaluableColumn'],
                task_definition['ColdStartColumn'], dataset, task,
                candidate_policy, relevance_name,
            )
            metrics.insert(4, 'Model', 'learner_neighbor_cf')
            metrics['NeighborCount'] = selected_neighbors
            metrics['NeighborQueryRuntimeSeconds'] = neighbor_query_runtime
            metrics['RecommendationBuildRuntimeSeconds'] = recommendation_runtime
            metrics['TrainingLearnerCohort'] = 'discovery'
            neighbor_metric_parts.append(metrics)

            output = recommendations.copy()
            output.insert(0, 'RelevanceDefinition', relevance_name)
            output.insert(0, 'CandidatePolicy', candidate_policy)
            output.insert(0, 'Model', 'learner_neighbor_cf')
            output.insert(0, 'Task', task)
            output.insert(0, 'Dataset', dataset)
            output['NeighborCount'] = selected_neighbors
            neighbor_recommendation_parts.append(output)

neighbor_tuning_metrics = pd.concat(neighbor_tuning_parts, ignore_index=True)
neighbor_selection = pd.DataFrame(neighbor_selection_rows)
neighbor_metrics = pd.concat(neighbor_metric_parts, ignore_index=True)
neighbor_recommendations = pd.concat(
    neighbor_recommendation_parts, ignore_index=True
)
assert neighbor_selection[['Dataset', 'Task']].drop_duplicates().shape[0] == 4
display(neighbor_selection)
display(neighbor_tuning_metrics.sort_values(
    ['Dataset', 'Task', 'NeighborCount', 'Segment']
))
display(neighbor_metrics.sort_values(
    ['Dataset', 'Task', 'RelevanceDefinition', 'CandidatePolicy', 'Segment', 'K']
))

,Dataset,Task,SelectedNeighbors,SelectionMetric,SelectionK,SelectionRelevance,SelectionCandidatePolicy,DiscoveryTrainingLearners,DiscoveryTuningLearners,ValidationLearnersUntouched
0,ASSISTments,problem,20,NDCGAtK,10,attempted,all_supported,21334,5334,6667
1,ASSISTments,skill,20,NDCGAtK,10,attempted,all_supported,21334,5334,6667
2,KDD,problem,20,NDCGAtK,10,attempted,all_supported,361,91,113
3,KDD,skill,20,NDCGAtK,10,attempted,all_supported,361,91,113


,Dataset,Task,CandidatePolicy,RelevanceDefinition,Model,Segment,K,PrecisionAtK,RecallAtK,NDCGAtK,...,CatalogCoverageAtK,MeanRecommendations,EvaluatedLearners,EvaluableRate,EvaluationScope,RecommendationMemoryBytes,MetricRuntimeSeconds,NeighborCount,ActualNeighborCount,TuningSource
0,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_tuning,all,10,0.440946,0.265667,0.498437,...,0.194474,9.802667,4799,0.899700,provided_learner_ids,13023370,2.267527,20,20,discovery_only
1,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_tuning,cold_start,10,0.000000,0.000000,0.000000,...,0.000000,0.000000,13,0.899700,provided_learner_ids,13023370,2.267527,20,20,discovery_only
2,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_tuning,non_cold_start,10,0.442144,0.266388,0.499790,...,0.194474,9.829294,4786,0.899700,provided_learner_ids,13023370,2.267527,20,20,discovery_only
3,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_tuning,all,10,0.421671,0.252298,0.475364,...,0.145001,9.864972,4799,0.899700,provided_learner_ids,13369101,2.225487,50,50,discovery_only
4,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_tuning,cold_start,10,0.000000,0.000000,0.000000,...,0.000000,0.000000,13,0.899700,provided_learner_ids,13369101,2.225487,50,50,discovery_only
5,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_tuning,non_cold_start,10,0.422817,0.252983,0.476655,...,0.145001,9.891768,4786,0.899700,provided_learner_ids,13369101,2.225487,50,50,discovery_only
6,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_tuning,all,10,0.403522,0.239419,0.454653,...,0.116544,9.930402,4799,0.899700,provided_learner_ids,13540074,3.401763,100,100,discovery_only
7,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_tuning,cold_start,10,0.000000,0.000000,0.000000,...,0.000000,0.000000,13,0.899700,provided_learner_ids,13540074,3.401763,100,100,discovery_only
8,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_tuning,non_cold_start,10,0.404618,0.240070,0.455888,...,0.116544,9.957376,4786,0.899700,provided_learner_ids,13540074,3.401763,100,100,discovery_only
9,ASSISTments,skill,all_supported,attempted,learner_neighbor_cf_tuning,all,10,0.460660,0.685504,0.737441,...,0.962733,9.021161,3119,0.584739,provided_learner_ids,6720201,1.590378,20,20,discovery_only


,Dataset,Task,CandidatePolicy,RelevanceDefinition,Model,Segment,K,PrecisionAtK,RecallAtK,NDCGAtK,...,MeanRecommendations,EvaluatedLearners,EvaluableRate,EvaluationScope,RecommendationMemoryBytes,MetricRuntimeSeconds,NeighborCount,NeighborQueryRuntimeSeconds,RecommendationBuildRuntimeSeconds,TrainingLearnerCohort
0,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf,all,5,0.479520,0.172009,0.509090,...,4.949300,5996,0.899355,validation,16107187,6.252257,20,4.204674,1.639634,discovery
1,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf,all,10,0.439626,0.270136,0.498392,...,9.796197,5996,0.899355,validation,16107187,6.252257,20,4.204674,1.639634,discovery
2,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf,all,20,0.363000,0.373978,0.495385,...,19.073549,5996,0.899355,validation,16107187,6.252257,20,4.204674,1.639634,discovery
3,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf,cold_start,5,0.000000,0.000000,0.000000,...,0.000000,21,0.899355,validation,16107187,6.252257,20,4.204674,1.639634,discovery
4,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf,cold_start,10,0.000000,0.000000,0.000000,...,0.000000,21,0.899355,validation,16107187,6.252257,20,4.204674,1.639634,discovery
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,KDD,skill,novel_only,successful,learner_neighbor_cf,all,10,0.402273,0.784812,0.732036,...,9.840909,88,0.778761,validation,384357,0.070141,20,0.015586,0.014677,discovery
116,KDD,skill,novel_only,successful,learner_neighbor_cf,all,20,0.255114,0.919970,0.759196,...,17.875000,88,0.778761,validation,384357,0.070141,20,0.015586,0.014677,discovery
117,KDD,skill,novel_only,successful,learner_neighbor_cf,non_cold_start,5,0.529545,0.586122,0.679687,...,4.965909,88,0.778761,validation,384357,0.070141,20,0.015586,0.014677,discovery
118,KDD,skill,novel_only,successful,learner_neighbor_cf,non_cold_start,10,0.402273,0.784812,0.732036,...,9.840909,88,0.778761,validation,384357,0.070141,20,0.015586,0.014677,discovery


## Step 10 — Cluster-based collaborative-filtering ablations

The selected Step 9 neighbour count is reused for two cluster-aware comparisons:

- `same_cluster_neighbor_cf`: neighbours are discovery learners from the query learner's cluster only.
- `cluster_popularity`: items are ranked by discovery-future relevance within each cluster.

ASSISTments uses its accepted clustering solution for comparison. Every KDD cluster result is labelled `exploratory_ablation_only` and cannot determine the production method.

In [16]:
def build_cluster_neighbor_cache(
    learners, early_history, catalog, selected_neighbors,
):
    item_order = sorted(catalog['item_id'].astype(str).unique())
    discovery = learners[learners['cohort'].eq('discovery')].copy()
    validation = learners[learners['cohort'].eq('validation')].copy()
    cache = []
    for cluster_value in sorted(validation['cluster'].unique(), key=str):
        training_ids = sorted(
            discovery.loc[discovery['cluster'].eq(cluster_value), 'learner_id']
            .astype(str)
        )
        query_ids = sorted(
            validation.loc[validation['cluster'].eq(cluster_value), 'learner_id']
            .astype(str)
        )
        if not query_ids or not training_ids:
            continue
        assert set(training_ids).isdisjoint(query_ids)
        training_matrix = build_sparse_history_matrix(
            early_history, training_ids, item_order
        )
        query_matrix = build_sparse_history_matrix(
            early_history, query_ids, item_order
        )
        indices, similarities = query_sparse_neighbors(
            training_matrix, query_matrix, selected_neighbors
        )
        cache.append({
            'cluster': cluster_value,
            'training_ids': training_ids,
            'query_ids': query_ids,
            'neighbor_indices': indices,
            'neighbor_similarities': similarities,
        })
    return cache, item_order

def same_cluster_recommendations(
    cache, item_order, relevance, early_history, relevance_column,
    candidate_policy,
):
    parts = []
    for entry in cache:
        target_matrix = build_discovery_target_matrix(
            relevance, entry['training_ids'], item_order, relevance_column
        )
        part = aggregate_neighbor_recommendations(
            entry['query_ids'], target_matrix, entry['neighbor_indices'],
            entry['neighbor_similarities'], item_order, early_history,
            candidate_policy,
        )
        parts.append(part)
    nonempty = [part for part in parts if not part.empty]
    if not nonempty:
        return pd.DataFrame(columns=['learner_id', 'item_id', 'score', 'rank'])
    recommendations = pd.concat(nonempty, ignore_index=True)
    assert not recommendations.duplicated(['learner_id', 'item_id']).any()
    return recommendations

def discovery_cluster_popularity_scores(
    relevance, learners, catalog, relevance_column,
):
    discovery = learners[learners['cohort'].eq('discovery')][
        ['learner_id', 'cluster']
    ].copy()
    validation_ids = set(
        learners.loc[learners['cohort'].eq('validation'), 'learner_id'].astype(str)
    )
    labels = relevance[
        relevance['learner_id'].astype(str).isin(
            set(discovery['learner_id'].astype(str))
        )
        & relevance['in_candidate_catalog']
        & relevance[relevance_column].eq(1)
    ][['learner_id', 'item_id']].drop_duplicates().merge(
        discovery, on='learner_id', how='inner', validate='many_to_one'
    )
    assert set(labels['learner_id'].astype(str)).isdisjoint(validation_ids)
    counts = labels.groupby(['cluster', 'item_id'])['learner_id'].nunique().rename(
        'discovery_cluster_relevant_learners'
    ).reset_index()
    clusters = sorted(learners['cluster'].unique(), key=str)
    complete_index = pd.MultiIndex.from_product(
        [clusters, sorted(catalog['item_id'].astype(str).unique())],
        names=['cluster', 'item_id'],
    ).to_frame(index=False)
    scores = complete_index.merge(
        counts, on=['cluster', 'item_id'], how='left', validate='one_to_one'
    )
    scores['discovery_cluster_relevant_learners'] = (
        scores['discovery_cluster_relevant_learners'].fillna(0).astype('int64')
    )
    scores['score'] = scores['discovery_cluster_relevant_learners'].astype(float)
    cluster_totals = scores.groupby('cluster')['score'].transform('sum')
    scores['normalised_score'] = 0.0
    positive_total = cluster_totals.gt(0)
    scores.loc[positive_total, 'normalised_score'] = (
        scores.loc[positive_total, 'score']
        / cluster_totals.loc[positive_total]
    )
    scores['score_source'] = f'discovery_cluster_future_{relevance_column}'
    return scores

def cluster_popularity_recommendations(
    learners, cluster_scores, early_history, candidate_policy,
):
    validation = learners[learners['cohort'].eq('validation')]
    parts = []
    for cluster_value, cluster_learners in validation.groupby('cluster'):
        item_scores = cluster_scores[
            cluster_scores['cluster'].eq(cluster_value)
        ][['item_id', 'score']]
        part = global_top_k_recommendations(
            cluster_learners['learner_id'], item_scores, early_history,
            candidate_policy,
        )
        parts.append(part)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(
        columns=['learner_id', 'item_id', 'score', 'rank']
    )

In [17]:
cluster_metric_parts = []
cluster_recommendation_parts = []
cluster_popularity_score_parts = []
cluster_diagnostic_rows = []

for task_definition in TASK_DEFINITIONS:
    dataset = task_definition['Dataset']
    task = task_definition['Task']
    tables = load_task_tables(task_definition)
    learners = tables['learners']
    catalog = tables['catalog']
    early_history = tables['early_history']
    future_relevance = tables['future_relevance']
    selected_neighbors = int(neighbor_selection.loc[
        neighbor_selection['Dataset'].eq(dataset)
        & neighbor_selection['Task'].eq(task),
        'SelectedNeighbors',
    ].iloc[0])
    evidence_use = (
        'accepted_ablation' if dataset == 'ASSISTments'
        else 'exploratory_ablation_only'
    )

    cache_started = time.perf_counter()
    cluster_cache, item_order = build_cluster_neighbor_cache(
        learners, early_history, catalog, selected_neighbors
    )
    cache_runtime = time.perf_counter() - cache_started
    covered_validation_ids = set().union(*(
        set(entry['query_ids']) for entry in cluster_cache
    )) if cluster_cache else set()
    expected_validation_ids = set(
        learners.loc[learners['cohort'].eq('validation'), 'learner_id'].astype(str)
    )
    cluster_diagnostic_rows.append({
        'Dataset': dataset,
        'Task': task,
        'ClusterEvidenceUse': evidence_use,
        'ClustersWithTrainingAndValidationLearners': len(cluster_cache),
        'ValidationLearnersCoveredByClusterNeighbors': len(covered_validation_ids),
        'ValidationLearners': len(expected_validation_ids),
        'ClusterNeighborCacheRuntimeSeconds': cache_runtime,
    })

    for relevance_name, relevance_column in RELEVANCE_DEFINITIONS.items():
        cluster_popularity_scores = discovery_cluster_popularity_scores(
            future_relevance, learners, catalog, relevance_column
        )
        score_output = cluster_popularity_scores.copy()
        score_output.insert(0, 'RelevanceDefinition', relevance_name)
        score_output.insert(0, 'Model', 'cluster_popularity')
        score_output.insert(0, 'Task', task)
        score_output.insert(0, 'Dataset', dataset)
        score_output['ClusterEvidenceUse'] = evidence_use
        cluster_popularity_score_parts.append(score_output)

        for candidate_policy in CANDIDATE_POLICIES:
            same_cluster_started = time.perf_counter()
            same_cluster = same_cluster_recommendations(
                cluster_cache, item_order, future_relevance, early_history,
                relevance_column, candidate_policy,
            )
            same_cluster_runtime = time.perf_counter() - same_cluster_started
            cluster_popularity_started = time.perf_counter()
            cluster_popularity = cluster_popularity_recommendations(
                learners, cluster_popularity_scores, early_history,
                candidate_policy,
            )
            cluster_popularity_runtime = (
                time.perf_counter() - cluster_popularity_started
            )
            models = {
                'same_cluster_neighbor_cf': (
                    same_cluster, same_cluster_runtime
                ),
                'cluster_popularity': (
                    cluster_popularity, cluster_popularity_runtime
                ),
            }
            for model_name, (recommendations, recommendation_runtime) in models.items():
                assert_candidate_subset(recommendations, catalog)
                metrics = evaluate_recommendations(
                    recommendations, future_relevance, catalog, learners,
                    relevance_column, task_definition['EvaluableColumn'],
                    task_definition['ColdStartColumn'], dataset, task,
                    candidate_policy, relevance_name,
                )
                metrics.insert(4, 'Model', model_name)
                metrics['NeighborCount'] = (
                    selected_neighbors
                    if model_name == 'same_cluster_neighbor_cf' else np.nan
                )
                metrics['ClusterEvidenceUse'] = evidence_use
                metrics['RecommendationBuildRuntimeSeconds'] = recommendation_runtime
                cluster_metric_parts.append(metrics)

                output = recommendations.copy()
                output.insert(0, 'RelevanceDefinition', relevance_name)
                output.insert(0, 'CandidatePolicy', candidate_policy)
                output.insert(0, 'Model', model_name)
                output.insert(0, 'Task', task)
                output.insert(0, 'Dataset', dataset)
                output['ClusterEvidenceUse'] = evidence_use
                cluster_recommendation_parts.append(output)

cluster_metrics = pd.concat(cluster_metric_parts, ignore_index=True)
cluster_recommendations = pd.concat(
    cluster_recommendation_parts, ignore_index=True
)
cluster_popularity_scores = pd.concat(
    cluster_popularity_score_parts, ignore_index=True
)
cluster_diagnostics = pd.DataFrame(cluster_diagnostic_rows)
cf_comparison_metrics = pd.concat(
    [neighbor_metrics, cluster_metrics], ignore_index=True, sort=False
)
assert set(cf_comparison_metrics['Model']) == {
    'learner_neighbor_cf', 'same_cluster_neighbor_cf', 'cluster_popularity'
}
assert cluster_metrics.loc[
    cluster_metrics['Dataset'].eq('KDD'), 'ClusterEvidenceUse'
].eq('exploratory_ablation_only').all()
display(cluster_diagnostics)
display(cf_comparison_metrics.sort_values(
    ['Dataset', 'Task', 'RelevanceDefinition', 'CandidatePolicy',
     'Segment', 'K', 'Model']
))

,Dataset,Task,ClusterEvidenceUse,ClustersWithTrainingAndValidationLearners,ValidationLearnersCoveredByClusterNeighbors,ValidationLearners,ClusterNeighborCacheRuntimeSeconds
0,ASSISTments,problem,accepted_ablation,2,6667,6667,8.887375
1,ASSISTments,skill,accepted_ablation,2,6667,6667,3.403690
2,KDD,problem,exploratory_ablation_only,3,113,113,0.130086
3,KDD,skill,exploratory_ablation_only,3,113,113,0.126103


,Dataset,Task,CandidatePolicy,RelevanceDefinition,Model,Segment,K,PrecisionAtK,RecallAtK,NDCGAtK,...,EvaluatedLearners,EvaluableRate,EvaluationScope,RecommendationMemoryBytes,MetricRuntimeSeconds,NeighborCount,NeighborQueryRuntimeSeconds,RecommendationBuildRuntimeSeconds,TrainingLearnerCohort,ClusterEvidenceUse
129,ASSISTments,problem,all_supported,attempted,cluster_popularity,all,5,0.031621,0.009281,0.031650,...,5996,0.899355,validation,17866552,5.535097,NaN,NaN,0.287884,NaN,accepted_ablation
0,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf,all,5,0.479520,0.172009,0.509090,...,5996,0.899355,validation,16107187,6.252257,20.0,4.204674,1.639634,discovery,NaN
120,ASSISTments,problem,all_supported,attempted,same_cluster_neighbor_cf,all,5,0.472515,0.167946,0.500354,...,5996,0.899355,validation,16111002,4.633735,20.0,NaN,3.162885,NaN,accepted_ablation
130,ASSISTments,problem,all_supported,attempted,cluster_popularity,all,10,0.031004,0.017859,0.031371,...,5996,0.899355,validation,17866552,5.535097,NaN,NaN,0.287884,NaN,accepted_ablation
1,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf,all,10,0.439626,0.270136,0.498392,...,5996,0.899355,validation,16107187,6.252257,20.0,4.204674,1.639634,discovery,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,KDD,skill,novel_only,successful,learner_neighbor_cf,non_cold_start,10,0.402273,0.784812,0.732036,...,88,0.778761,validation,384357,0.070141,20.0,0.015586,0.014677,discovery,NaN
352,KDD,skill,novel_only,successful,same_cluster_neighbor_cf,non_cold_start,10,0.361364,0.681684,0.657783,...,88,0.778761,validation,408671,0.162981,20.0,NaN,0.110972,NaN,exploratory_ablation_only
359,KDD,skill,novel_only,successful,cluster_popularity,non_cold_start,20,0.209091,0.773370,0.573659,...,88,0.778761,validation,462737,0.191659,NaN,NaN,0.065885,NaN,exploratory_ablation_only
119,KDD,skill,novel_only,successful,learner_neighbor_cf,non_cold_start,20,0.255114,0.919970,0.759196,...,88,0.778761,validation,384357,0.070141,20.0,0.015586,0.014677,discovery,NaN


## Save and verify Steps 8–10 artifacts

In [18]:
steps_8_10_metrics = pd.concat(
    [content_metrics, neighbor_metrics, cluster_metrics],
    ignore_index=True, sort=False,
)
steps_8_10_recommendations = pd.concat(
    [content_recommendations, neighbor_recommendations, cluster_recommendations],
    ignore_index=True, sort=False,
)

steps_8_10_paths = {
    'metrics': OUTPUT_ROOT / 'steps_8_10_metrics.csv',
    'recommendations': OUTPUT_ROOT / 'steps_8_10_recommendations.parquet',
    'content_scores': OUTPUT_ROOT / 'step8_content_scores.parquet',
    'neighbor_tuning': OUTPUT_ROOT / 'step9_neighbor_tuning.csv',
    'neighbor_selection': OUTPUT_ROOT / 'step9_neighbor_selection.csv',
    'cluster_popularity_scores': OUTPUT_ROOT / 'step10_cluster_popularity_scores.parquet',
    'cluster_diagnostics': OUTPUT_ROOT / 'step10_cluster_diagnostics.csv',
    'config': OUTPUT_ROOT / 'steps_8_10_config.json',
}
steps_8_10_metrics.to_csv(steps_8_10_paths['metrics'], index=False)
steps_8_10_recommendations.to_parquet(
    steps_8_10_paths['recommendations'], index=False, compression='snappy'
)
content_scores.to_parquet(
    steps_8_10_paths['content_scores'], index=False, compression='snappy'
)
neighbor_tuning_metrics.to_csv(
    steps_8_10_paths['neighbor_tuning'], index=False
)
neighbor_selection.to_csv(
    steps_8_10_paths['neighbor_selection'], index=False
)
cluster_popularity_scores.to_parquet(
    steps_8_10_paths['cluster_popularity_scores'],
    index=False, compression='snappy',
)
cluster_diagnostics.to_csv(
    steps_8_10_paths['cluster_diagnostics'], index=False
)

steps_8_10_config = {
    'implemented_steps': [8, 9, 10],
    'content_weights': CONTENT_WEIGHTS,
    'content_popularity_tie_break_weight': POPULARITY_TIE_BREAK_WEIGHT,
    'content_profile_source': 'validation_early_only',
    'content_catalog_and_mapping_source': 'discovery_early_only',
    'neighbor_candidates': list(NEIGHBOR_CANDIDATES),
    'neighbor_selection_metric': 'attempted_all_supported_NDCGAt10',
    'neighbor_tuning_source': 'discovery_only',
    'neighbor_selection': neighbor_selection.to_dict(orient='records'),
    'neighbor_similarity': 'cosine_on_log1p_early_interaction_counts',
    'neighbor_target_source': 'discovery_future_relevance_only',
    'cluster_models': ['same_cluster_neighbor_cf', 'cluster_popularity'],
    'cluster_role': 'ablation_only',
    'kdd_cluster_role': 'exploratory_ablation_only',
    'candidate_policies': list(CANDIDATE_POLICIES),
    'relevance_definitions': RELEVANCE_DEFINITIONS,
    'maximum_saved_recommendations_per_learner': MAX_RECOMMENDATIONS,
    'dense_learner_item_matrix_constructed': False,
    'validation_future_role': 'evaluation_only',
}
with open(steps_8_10_paths['config'], 'w', encoding='utf-8') as file:
    json.dump(steps_8_10_config, file, indent=2)

steps_8_10_manifest = pd.DataFrame([
    {
        'Artifact': name,
        'File': path.name,
        'Rows': (
            pq.ParquetFile(path).metadata.num_rows
            if path.suffix == '.parquet'
            else (len(pd.read_csv(path)) if path.suffix == '.csv' else 1)
        ),
        'Bytes': path.stat().st_size,
    }
    for name, path in steps_8_10_paths.items()
])
steps_8_10_manifest.to_csv(
    OUTPUT_ROOT / 'steps_8_10_artifact_manifest.csv', index=False
)
assert len(steps_8_10_manifest) == 8
assert pq.ParquetFile(
    steps_8_10_paths['recommendations']
).metadata.num_rows == len(steps_8_10_recommendations)
assert len(pd.read_csv(steps_8_10_paths['metrics'])) == len(steps_8_10_metrics)
assert len(pd.read_csv(steps_8_10_paths['neighbor_selection'])) == 4
display(steps_8_10_manifest)

,Artifact,File,Rows,Bytes
0,metrics,steps_8_10_metrics.csv,420,124249
1,recommendations,steps_8_10_recommendations.parquet,2825777,11900094
2,content_scores,step8_content_scores.parquet,6770282,145442477
3,neighbor_tuning,step9_neighbor_tuning.csv,30,8612
4,neighbor_selection,step9_neighbor_selection.csv,4,446
5,cluster_popularity_scores,step10_cluster_popularity_scores.parquet,165484,948671
6,cluster_diagnostics,step10_cluster_diagnostics.csv,4,441
7,config,steps_8_10_config.json,1,2686
